# Advanced Retrieval for RAG

Naive **vector search** is a strong baseline — but production RAG often needs more. Exact error codes, SKU strings, and rare tokens may not rank highly in embedding space. Broad questions return redundant Alembic chunks (**c2** and **c5**). Wrong document types pollute answers when the corpus mixes API, security, and test docs.

**Advanced retrieval** layers techniques on top of bi-encoder search: **metadata filters**, **hybrid lexical + semantic** fusion, **reranking**, **MMR diversity**, and **multi-query** recall. These attack different failure modes *before* context assembly (**06**) and prompting (**05**).

This notebook implements hands-on patterns with **in-memory Chroma**, OpenAI embeddings, and NumPy helpers on the shared **c1–c5** corpus. Vector DB vendor comparisons and ANN tuning remain in **05. Embeddings & Vector DBs** (**07** metadata filters, **08** landscape).

Prerequisites: **06. Context Assembly and Window Management**, **05.02 Similarity Search**, **05.07 Metadata Filters**, **03. RAG Architecture Patterns** (query transform overview). Next: **08. RAG Failure Modes and Grounding**.

In [ ]:
OPENAI_API_KEY = "sk-proj-placeholder-replace-with-your-real-key"

import chromadb
import numpy as np
from openai import OpenAI

openai_client = OpenAI(api_key=OPENAI_API_KEY)
EMBED_MODEL = "text-embedding-3-small"
CHAT_MODEL = "gpt-4o-mini"

np.set_printoptions(precision=4, suppress=True)

DOCUMENTS = [
    {
        "id": "c1",
        "text": (
            "The Notes API is built with FastAPI and stores notes in memory for demos. "
            "Routes live under /notes with GET, POST, PUT, and DELETE handlers."
        ),
        "source": "api-docs",
    },
    {
        "id": "c2",
        "text": (
            "Alembic applies SQLAlchemy schema migrations. "
            "Run alembic upgrade head after pulling new revisions."
        ),
        "source": "db-docs",
    },
    {
        "id": "c3",
        "text": (
            "JWT bearer tokens authenticate API requests. "
            "Clients send Authorization: Bearer token header."
        ),
        "source": "security-docs",
    },
    {
        "id": "c4",
        "text": (
            "Pytest fixtures share database setup across tests. "
            "Use conftest.py for session-scoped engines and dependency overrides."
        ),
        "source": "test-docs",
    },
    {
        "id": "c5",
        "text": (
            "Alembic autogenerate compares SQLAlchemy models to the live database schema "
            "and drafts revision files."
        ),
        "source": "db-docs",
    },
]


def embed_texts(texts: list[str]) -> list[list[float]]:
    response = openai_client.embeddings.create(model=EMBED_MODEL, input=texts)
    ordered = sorted(response.data, key=lambda item: item.index)
    return [item.embedding for item in ordered]


def fresh_collection(name: str = "advanced_rag_lesson"):
    client = chromadb.Client()
    try:
        client.delete_collection(name)
    except Exception:
        pass
    return client.create_collection(name=name, metadata={"hnsw:space": "cosine"})


print("Setup OK")

---

## 1. When Vector Search Alone Fails

| Query type | Symptom | Advanced technique |
|------------|---------|-------------------|
| Exact token `conftest` | Semantic search ranks generic pytest text lower | **Hybrid** BM25 + vector |
| Error code `401` / SKU | Rare token weak in embeddings | Keyword boost |
| "database schema" | Returns only c2, misses c5 nuance | **Rerank** or higher k |
| Broad Alembic question | c2 + c5 redundant | **MMR** diversity |
| Security question | c1 API noise in results | **Metadata filter** on `source` |
| Paraphrased question | Miss despite good embeddings | **Multi-query** / HyDE (**03**) |

```
Naive:  query ──► embed ──► top-k
Advanced: query ──► [filter | hybrid | multi-query] ──► top-N ──► [rerank | MMR] ──► top-k
```

Add complexity only when **05.09** retrieval eval shows a specific miss pattern.

---

## 2. Index the Corpus with Metadata

Advanced retrieval depends on **metadata** ingested at index time (**04**). Each chunk carries a `source` tag for filtering and routing.

In [ ]:
collection = fresh_collection()
for d in DOCUMENTS:
    collection.add(
        ids=[d["id"]],
        documents=[d["text"]],
        embeddings=embed_texts([d["text"]]),
        metadatas=[{"source": d["source"]}],
    )
print(f"Indexed {collection.count()} chunks")
for d in DOCUMENTS:
    print(f"  {d['id']}  {d['source']}")

---

## 3. Metadata Filtering

Restrict search to a **subset** of the index — by document type, tenant, ACL, or product.

```python
collection.query(
    query_embeddings=[q_vec],
    n_results=k,
    where={"source": "security-docs"},
)
```

### 3.1 When to Filter

| Signal | Filter example |
|--------|----------------|
| User on security FAQ page | `source=security-docs` |
| Multi-tenant SaaS | `tenant_id=acme` |
| Role-based access | `acl=internal` (**04**) |
| Router from **03** | Map intent → allowed sources |

Filtering reduces noise **before** similarity scoring — cheaper than reranking irrelevant corpora.

In [ ]:
def retrieve_semantic(query: str, k: int = 5, source: str | None = None) -> dict:
    q_emb = embed_texts([query])[0]
    kwargs = dict(
        query_embeddings=[q_emb],
        n_results=k,
        include=["documents", "metadatas", "distances", "ids"],
    )
    if source:
        kwargs["where"] = {"source": source}
    return collection.query(**kwargs)


q = "How are API requests authenticated?"
print("=== Unfiltered ===")
hits = retrieve_semantic(q, k=3)
for cid, doc, meta in zip(hits["ids"][0], hits["documents"][0], hits["metadatas"][0]):
    print(f"  {cid}  {meta['source']:15s}  {doc[:55]}...")

print("\n=== Filter: security-docs ===")
hits_f = retrieve_semantic(q, k=3, source="security-docs")
for cid, doc in zip(hits_f["ids"][0], hits_f["documents"][0]):
    print(f"  {cid}  {doc[:60]}...")

---

## 4. Hybrid Retrieval — Lexical + Semantic

**Bi-encoder** vector search handles paraphrase. **Lexical** search (BM25, keyword overlap) handles exact tokens. **Hybrid** combines both.

```
              ┌── semantic ranks (embedding similarity)
Query ────────┤
              └── lexical ranks (BM25 / keyword overlap)
                          │
                          ▼
                   fuse scores (RRF or weighted sum)
```

### 4.1 Reciprocal Rank Fusion (RRF)

RRF merges ranked lists without normalizing incompatible scores:

$$\text{RRF}(d) = \sum_{r \in \text{retrievers}} \frac{1}{k_{rrf} + \text{rank}_r(d)}$$

Common $k_{rrf} = 60$. Document $d$ appearing high on **both** lists gets a boosted fused score.

In [ ]:
def keyword_overlap_score(query: str, doc: str) -> float:
    q_tokens = set(query.lower().split())
    d_tokens = set(doc.lower().split())
    if not q_tokens:
        return 0.0
    return len(q_tokens & d_tokens) / len(q_tokens)


def rrf_fuse(rank_lists: list[list[str]], k_rrf: int = 60) -> list[tuple[str, float]]:
    scores: dict[str, float] = {}
    for ranked_ids in rank_lists:
        for rank, doc_id in enumerate(ranked_ids, start=1):
            scores[doc_id] = scores.get(doc_id, 0.0) + 1.0 / (k_rrf + rank)
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)


query = "alembic upgrade migration"
id_to_doc = {d["id"]: d["text"] for d in DOCUMENTS}

semantic_hits = retrieve_semantic(query, k=5)
semantic_rank = semantic_hits["ids"][0]

lexical_rank = sorted(
    DOCUMENTS,
    key=lambda d: keyword_overlap_score(query, d["text"]),
    reverse=True,
)
lexical_ids = [d["id"] for d in lexical_rank]

fused = rrf_fuse([semantic_rank, lexical_ids])
print(f"Query: {query}\n")
print("Semantic top-3:", semantic_rank[:3])
print("Lexical top-3: ", lexical_ids[:3])
print("RRF fused:     ", fused[:3])

---

## 5. Weighted Score Fusion (Alternative)

When raw scores are comparable, use a weighted sum:

$$\text{score}(d) = \alpha \cdot \text{sim}_{\text{semantic}}(q, d) + (1 - \alpha) \cdot \text{sim}_{\text{lexical}}(q, d)$$

| $\alpha$ | Bias |
|----------|------|
| 0.8 | Mostly semantic (default RAG) |
| 0.5 | Balanced |
| 0.3 | Keyword-heavy (error codes, SKUs) |

Tune $\alpha$ on a labeled dev set — no universal constant.

In [ ]:
def cosine_sim(a: list[float], b: list[float]) -> float:
    va, vb = np.array(a), np.array(b)
    return float(np.dot(va, vb) / (np.linalg.norm(va) * np.linalg.norm(vb) + 1e-9))


def hybrid_weighted(query: str, alpha: float = 0.7) -> list[tuple[str, float]]:
    q_emb = embed_texts([query])[0]
    doc_embs = embed_texts([d["text"] for d in DOCUMENTS])
    scored = []
    for d, emb in zip(DOCUMENTS, doc_embs):
        sem = cosine_sim(q_emb, emb)
        lex = keyword_overlap_score(query, d["text"])
        combined = alpha * sem + (1 - alpha) * lex
        scored.append((d["id"], combined, sem, lex))
    scored.sort(key=lambda x: x[1], reverse=True)
    return scored


q = "pytest conftest fixtures"
for cid, combined, sem, lex in hybrid_weighted(q)[:4]:
    print(f"{cid}  combined={combined:.3f}  sem={sem:.3f}  lex={lex:.3f}")

Keyword overlap boosts **c4** when the query names `conftest` explicitly — even if semantic rank was ambiguous.

---

## 6. Retrieve Broad, Then Narrow

Advanced pipelines use a **two-stage** width:

| Stage | k | Purpose |
|-------|---|--------|
| **Candidate generation** | 20–100 | High recall |
| **Rerank / MMR / assemble** | 3–8 | High precision (**06**) |

```
query ──► retrieve(k=30) ──► rerank ──► MMR ──► assemble(budget) ──► prompt
```

Never rerank the entire corpus — only **candidates** from stage one.

---

## 7. Reranking

| Model type | When it runs | Speed |
|------------|--------------|-------|
| **Bi-encoder** | Index + query embed | Fast |
| **Cross-encoder** | Each (query, chunk) pair | Slower, more accurate |
| **LLM grader** | Prompt: "rate relevance 1-5" | Flexible, costly |

### 7.1 When to Rerank

Add reranking when Recall@20 is good but **Precision@3** is poor — the answer chunk is *somewhere* in candidates but not at the top.

Production options: Cohere Rerank, `bge-reranker`, cross-encoder HuggingFace models. This notebook uses a **heuristic reranker** (blend semantic + keyword) when no cross-encoder is installed.

In [ ]:
def rerank_heuristic(query: str, candidate_ids: list[str], top_n: int = 3) -> list[str]:
    """Demo reranker: re-score candidates with hybrid weighted formula."""
    id_set = set(candidate_ids)
    candidates = [d for d in DOCUMENTS if d["id"] in id_set]
    q_emb = embed_texts([query])[0]
    scored = []
    for d in candidates:
        emb = embed_texts([d["text"]])[0]
        sem = cosine_sim(q_emb, emb)
        lex = keyword_overlap_score(query, d["text"])
        scored.append((d["id"], 0.6 * sem + 0.4 * lex))
    scored.sort(key=lambda x: x[1], reverse=True)
    return [cid for cid, _ in scored[:top_n]]


q = "Alembic autogenerate revision"
candidates = retrieve_semantic(q, k=5)["ids"][0]
reranked = rerank_heuristic(q, candidates, top_n=3)
print("Candidates (semantic):", candidates)
print("After rerank:         ", reranked)

---

## 8. Maximal Marginal Relevance (MMR)

When top semantic hits are **redundant** (c2 and c5 both about Alembic), **MMR** trades relevance for **diversity**:

$$\text{MMR} = \arg\max_{d \in D \setminus S}\; \lambda\,\text{sim}(q,d) - (1-\lambda)\,\max_{s \in S}\,\text{sim}(d,s)$$

| $\lambda$ | Behavior |
|-----------|----------|
| 1.0 | Pure relevance (no diversity) |
| 0.7 | Common starting point |
| 0.5 | Strong diversity — fewer near-duplicates |

Use MMR when assembly (**06**) would otherwise pack multiple overlapping Alembic chunks.

In [ ]:
def mmr_select(
    query_emb: list[float],
    doc_embs: list[list[float]],
    doc_ids: list[str],
    k: int = 3,
    lam: float = 0.7,
) -> list[str]:
    selected_idx: list[int] = []
    candidates = list(range(len(doc_ids)))
    while len(selected_idx) < k and candidates:
        best_i, best_score = None, -1e9
        for i in candidates:
            rel = cosine_sim(query_emb, doc_embs[i])
            div = max((cosine_sim(doc_embs[i], doc_embs[j]) for j in selected_idx), default=0.0)
            score = lam * rel - (1 - lam) * div
            if score > best_score:
                best_i, best_score = i, score
        selected_idx.append(best_i)
        candidates.remove(best_i)
    return [doc_ids[i] for i in selected_idx]


query = "Tell me about Alembic and database schema"
q_emb = embed_texts([query])[0]
all_ids = [d["id"] for d in DOCUMENTS]
all_embs = embed_texts([d["text"] for d in DOCUMENTS])

semantic_top = retrieve_semantic(query, k=5)["ids"][0]
mmr_top = mmr_select(q_emb, all_embs, all_ids, k=3, lam=0.6)

print("Semantic top-5:", semantic_top)
print("MMR top-3:     ", mmr_top)

MMR may prefer **c2** + **c1** over **c2** + **c5** — broader topic coverage for multi-faceted questions.

---

## 9. Multi-Query and Query Expansion

From notebook **03**: generate paraphrases or sub-queries, retrieve for each, **union** results, then dedupe / RRF.

```python
variants = [query, paraphrase_1, paraphrase_2]
all_ids = []
for v in variants:
    all_ids.extend(retrieve_semantic(v, k=5)["ids"][0])
union = dedupe_preserving_order(all_ids)
```

| Technique | Cost | Best for |
|-----------|------|----------|
| **Multi-query** | N embed calls | Paraphrase mismatch |
| **HyDE** | 1 LLM + 1 embed | Short question vs long answer docs |
| **Step-back** | 1 LLM | Need broader context first |

In [ ]:
def multi_query_union(queries: list[str], k: int = 3) -> list[str]:
    seen: list[str] = []
    for q in queries:
        ids = retrieve_semantic(q, k=k)["ids"][0]
        for cid in ids:
            if cid not in seen:
                seen.append(cid)
    return seen


variants = [
    "How do I run database migrations?",
    "alembic upgrade head SQLAlchemy",
    "apply schema revisions after git pull",
]
union = multi_query_union(variants, k=3)
print("Multi-query union:", union)

---

## 10. Composed `advanced_retrieve` Pipeline

Wire filter → semantic broad retrieval → rerank → MMR into one function.

In [ ]:
def advanced_retrieve(
    query: str,
    *,
    source: str | None = None,
    candidate_k: int = 5,
    final_k: int = 3,
    use_mmr: bool = True,
    mmr_lambda: float = 0.65,
) -> list[dict]:
    hits = retrieve_semantic(query, k=candidate_k, source=source)
    candidate_ids = hits["ids"][0]
    reranked_ids = rerank_heuristic(query, candidate_ids, top_n=candidate_k)

    id_to_doc = {d["id"]: d for d in DOCUMENTS}
    cand_docs = [id_to_doc[cid] for cid in reranked_ids if cid in id_to_doc]

    if use_mmr and len(cand_docs) > 1:
        q_emb = embed_texts([query])[0]
        embs = embed_texts([d["text"] for d in cand_docs])
        ids = [d["id"] for d in cand_docs]
        final_ids = mmr_select(q_emb, embs, ids, k=final_k, lam=mmr_lambda)
    else:
        final_ids = reranked_ids[:final_k]

    return [id_to_doc[cid] for cid in final_ids]


results = advanced_retrieve("What does Alembic autogenerate do?", source="db-docs")
for r in results:
    print(f"{r['id']}  {r['text'][:65]}...")

---

## 11. End-to-End `advanced_rag`

Connect advanced retrieval to prompting (**05**) and generation (**02**).

In [ ]:
GROUNDING_SYSTEM = """You answer ONLY from Context. If insufficient, say: I don't have that in the docs.
End with: Sources: <comma-separated chunk ids>"""


def format_context(chunks: list[dict]) -> str:
    parts = []
    for c in chunks:
        parts.append(f'<doc id="{c["id"]}" source="{c["source"]}">\n{c["text"]}\n</doc>')
    return "\n\n".join(parts)


def advanced_rag(question: str, source: str | None = None) -> str:
    chunks = advanced_retrieve(question, source=source, final_k=3)
    ctx = format_context(chunks)
    r = openai_client.chat.completions.create(
        model=CHAT_MODEL,
        temperature=0,
        messages=[
            {"role": "system", "content": GROUNDING_SYSTEM},
            {"role": "user", "content": f"Context:\n{ctx}\n\nQuestion: {question}"},
        ],
    )
    return r.choices[0].message.content


print(advanced_rag("What does Alembic autogenerate do?", source="db-docs"))

---

## 12. Technique Selection Guide

| If eval shows… | Try first |
|----------------|----------|
| Wrong document type in top-k | Metadata filter / router (**03**) |
| Keyword in query but semantic miss | Hybrid RRF |
| Right doc at rank 15 | Reranker + higher candidate_k |
| Redundant c2/c5 in context | MMR or dedupe (**06**) |
| Paraphrase miss | Multi-query / HyDE (**03**) |
| Still failing | Chunking + embed model (**05.04**, **05.03**) |

---

## 13. Tuning Knobs

| Knob | Starting value | Measure |
|------|----------------|--------|
| `candidate_k` | 10–20 | Recall@k (**05.09**) |
| `final_k` | 3–5 | Assembly budget (**06**) |
| MMR $\lambda$ | 0.65–0.75 | Diversity vs accuracy |
| RRF $k_{rrf}$ | 60 | Standard default |
| Hybrid $\alpha$ | 0.7 | Keyword-heavy query subset |
| Metadata routes | By `source` | Wrong-corpus rate |

---

## 14. Common Mistakes

| Mistake | Consequence | Fix |
|---------|-------------|-----|
| Hybrid without eval | Complexity, no gain | Measure on labeled queries |
| Rerank entire index | Latency explosion | Rerank candidates only |
| Filter too aggressive | Miss cross-source answers | Relax or multi-index search |
| MMR λ=0 always | Redundant chunks | Tune λ |
| Skip metadata at ingest | Filters useless | **04** schema |
| Fix retrieval when prompt fails | Wasted effort | Split eval layers (**09**) |

---

## 15. Debugging Advanced Retrieval

1. **Log candidate ids** before and after rerank/MMR
2. **Compare semantic-only vs hybrid** on failing queries
3. **Disable filter** — is metadata excluding the answer?
4. **Increase candidate_k** — is the chunk outside top-N?
5. **Inspect keyword scores** — does hybrid help?
6. **Return to chunking** if gold text never ranks in top-20

---

## 16. Summary

Advanced retrieval improves **what enters** the context window: **metadata filters** scope the search; **hybrid RRF** fuses lexical and semantic signals; **reranking** sharpens precision among candidates; **MMR** reduces redundancy; **multi-query** widens recall for paraphrases.

The pattern is **retrieve broad → narrow → assemble** (**06**) → **prompt** (**05**). Demonstrations indexed **c1–c5**, implemented RRF, weighted hybrid, heuristic rerank, MMR, multi-query union, `advanced_retrieve`, and `advanced_rag`.

Measure every addition on a labeled set (**05.09**, **09**) — complexity should earn its latency and cost.

Back: **06. Context Assembly and Window Management**. Next: **08. RAG Failure Modes and Grounding**.